In [ ]:
%pip install -q -U google-genai termcolor google-cloud-bigquery db-dtypes thefuzz jsonref google-cloud-aiplatform

In [ ]:
PROJECT_ID = "ghp-poc"
REGION = "us-central1"

DATA_AXLE = "ghp-poc.jobseq.data_axle"
INDUSTRY_2024Q3 = "ghp-poc.jobseq.industry_2024q3_cleaned"
INDUSTRY_OCCUPATION_2024Q3 = "ghp-poc.jobseq.industry_occupation_mix_2024q3"
OCCUPATION_WAGES_2024Q3 = "ghp-poc.jobseq.`jobseq-occupation-wages-2024q3`"

OCCUPATION_2024Q2 = "ghp-poc.jobseq.occupation_2024q2_cleaned"
WAGES_2024Q2 = "ghp-poc.jobseq.wages_2024q2"
NAICS_US_CODE_2022 = "ghp-poc.jobseq.naics_us_code_2022"

MAX_STORE_RESULTS = 10
MAX_PRODUCT_RESULTS = 10
STORE_NAME_SIMILARITY_THRESHOLD = 90
PRODUCT_SEARCH_SIMILARITY_THRESHOLD = 80

### Dependency Imports

In [ ]:
from typing import Optional, List, Tuple

import pydantic
import termcolor
import pandas as pd
from thefuzz import fuzz
from google import genai
from google.genai import types as genai_types
from google.cloud import bigquery

### Initialize Clients

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

### Pydantic Models

In [ ]:
# DB Models
class MetroMatrix(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]

class HQRelocation(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]
    Industry: Optional[str]

class CompanyRelocation(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]
    Industry: Optional[str]

# Tool return types
class MetroMatrixResult(pydantic.BaseModel):
    city_analysis: list[MetroMatrix] = []
    error: Optional[str] = None

class HQRelocationResult(pydantic.BaseModel):
    city_analysis: list[HQRelocation] = []
    error: Optional[str] = None

class CompanyRelocationResult(pydantic.BaseModel):
    city_analysis: list[CompanyRelocation] = []
    error: Optional[str] = None

## Tools

user journeys:
1. Metro Matrix request
    - City names
    - State names
2. Head Quarter Relocation
    - City name
    - State name
    - Industry name
3. Company Relocation
    - City name
    - State name
    - Industry name

### JobsEQ

In [879]:
def jobs_eq_data(
    city_names: List[str],
    economic_request_type: str,
    naics_codes: List[str],
    naics_titles: List[str],
    state_name: Optional[str]=None,
)->List[pd.DataFrame]:
    """
    Query BigQuery/JobsEQ data according to user request.

    Args:
        city_names: List of cities requested by user.
        economic_request_type: "metro_matrix", "hq_relocation", or "company_relocation".
        naics_codes: List of NAICS Codes requested for Industries.
        state_name: Optional, state specified by user with 2 digit representation.
    """

    if economic_request_type == "company_relocation":
        # Get the industry employment and wages by industry table.
        empl_and_wages_by_industry_table = empl_and_wages_by_industry(
            city_names=city_names,
            naics_codes=naics_codes,
            state_name=state_name,
        )
        unskilled_labor_salaries_by_job_table = unskilled_labor_wages(
            city_names=city_names,
            naics_codes=naics_codes,
            naics_titles=titles,
            state_name=state_name,
        )
        return empl_and_wages_by_industry_table, unskilled_labor_salaries_by_job_table
    elif economic_request_type == "hq_relocation":
        # Get the labor market information table.
        labor_market_info_table = labor_market_info(
            city_names=city_names,
            naics_codes=naics_codes,
            state_name=state_name,
        )
        return labor_market_info_table


### NAICS Code Mapping Function

In [901]:
! pip install Levenshtein


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [881]:
import re
import Levenshtein

def extract_naics_info(query, df):
    """
    Extracts NAICS codes and titles from a user query, handling misspellings and case sensitivity.

    Args:
        query (str): The user's query string.
        df (pd.DataFrame): The DataFrame containing NAICS codes and titles.

    Returns:
        tuple: A tuple containing two lists - one with NAICS titles and the other with corresponding NAICS codes.
    """

    # Compile regex patterns (case-insensitive)
    code_pattern = re.compile(r'\b(\d{2,6})\b')
    # Extract all unique NAICS titles from the dataframe
    titles = df['2022 NAICS US Title'].unique()

    # Find all potential codes in the query
    code_matches = code_pattern.findall(query)
    title_matches = []
    # Calculate Levenshtein distance for each title
    for title in titles:
        distance = Levenshtein.distance(query.lower(), title.lower())
        if distance <= 2:
            title_matches.append(title)

    found_titles = []
    found_codes = []
    found_pairs = set()  # Use a set to track (title, code) pairs

    # Process code matches
    for code in code_matches:
        # Find the row with the code
        row = df[df['code'] == code]
        # If the code is found, add the title and code to the lists
        if not row.empty:
            title = row.iloc[0]['2022 NAICS US Title']
            pair = (title, code)
            if pair not in found_pairs:
                found_titles.append(title)
                found_codes.append(code)
                found_pairs.add(pair)

    # Process title matches
    for title in title_matches:
        # Find all rows with the title
        rows = df[df['2022 NAICS US Title'] == title]
        # If the title is found, find the longest code and add the title and code to the lists
        if not rows.empty:
            longest_code = max(rows['code'], key=len)
            pair = (title, longest_code)
            if pair not in found_pairs:
                found_titles.append(title)
                found_codes.append(longest_code)
                found_pairs.add(pair)

    return found_titles, found_codes

In [902]:
csv_file_path = "naics_mappings.csv"
df = pd.read_csv(csv_file_path)

        code                           2022 NAICS US Title
0         11    Agriculture, Forestry, Fishing and Hunting
1        111                               Crop Production
2       1111                     Oilseed and Grain Farming
3      11111                               Soybean Farming
4     111110                               Soybean Farming
...      ...                                           ...
2120    9281  National Security and International Affairs 
2121   92811                            National Security 
2122  928110                            National Security 
2123   92812                        International Affairs 
2124  928120                        International Affairs 

[2125 rows x 2 columns]


In [903]:
! pip install pandas


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [900]:
# Example Usage
query = "I'm interested in NAICS codes 322"
titles, codes = extract_naics_info(query, df)
print("NAICS Titles:", titles)
print("NAICS Codes:", codes)

NAICS Titles: ['Paper Manufacturing']
NAICS Codes: ['322']


### Company Relocation Functions

In [884]:
def empl_and_wages_by_industry(
    city_names: List[str],
    naics_codes: List[str],
    state_name: Optional[str]=None,
)->pd.DataFrame:
    """
    Query BigQuery/JobsEQ data for the industry employment and wages by industry table for Company Relocation.

    Args:
        city_names: List of city's requested by user.
        naics_codes: List of NAICS Codes requested for Industries.
        state_name: Optional, state specified by user with 2 digit representation.

    Returns:
        Pandas dataframe representing the industry employment and wages by industry table.
    """

    # Get subsector numbers.
    naics_where_clause = f"(Substring(code, 1, {len(naics_codes[0])}) = '{naics_codes[0]}'"
    # Find the longest NAICS code to find the number substring.
    max_length = len(naics_codes[0])
    for code in naics_codes[1:]:
        naics_where_clause = naics_where_clause + f" OR SUBSTR(code, 1, {len(code)}) = '{code}'"
        max_length = max(max_length, len(code))
    naics_where_clause = naics_where_clause + f") AND LENGTH(code) = {max_length+1}"
    naics_sql = f"SELECT code, `2022 NAICS US Title` FROM {NAICS_US_CODE_2022} WHERE {naics_where_clause};"

    # Get child NAICS sector names and codes for Each parent sector.
    query_job = bq_client.query(
        query=naics_sql,
    )
    naics_dig: pd.DataFrame = query_job.to_dataframe()

    # For each child sector, calculate Employment and Current Avg Ann Wages for that city
    city_where_clause = f"(LOWER(metro) LIKE '%{city_names[0]}%'"
    if len(city_names)>1:
        for city in city_names[1:]:
            city_where_clause = city_where_clause + f" OR LOWER(metro) LIKE '%{city}%'"
    city_where_clause = city_where_clause + ")"

    sub_sector_sql = f"""
    SELECT
      SUBSTR(code, 1, {max_length+1}) AS code,
      SUM(empl) AS sum_total_empl_int,
      SUM(avg_ann_wages_int)/SUM(empl) AS avg_avg_ann_wages_int,
      metro
    FROM
      {INDUSTRY_2024Q3} t
    WHERE
      {city_where_clause}
      AND
      SUBSTR(code, 1, {max_length+1}) IN {tuple(naics_dig["code"])} Group by metro, code;"""

    query_job = bq_client.query(
        query=sub_sector_sql,
    )
    sub_sector_data: pd.DataFrame = query_job.to_dataframe()

    # Combine with naics 3 dig table
    merged_df = pd.merge(naics_dig, sub_sector_data, left_on='code', right_on='code', how='left')

    sector_employment_and_wages_by_sub_sector = merged_df[["2022 NAICS US Title", "sum_total_empl_int", "avg_avg_ann_wages_int", "metro"]]
    sector_employment_and_wages_by_sub_sector = sector_employment_and_wages_by_sub_sector.rename(columns = {
        "2022 NAICS US Title": "Industry",
        "sum_total_empl_int": "Employment",
        "avg_avg_ann_wages_int": "Current Avg Ann Wages",
        "metro": "Metro"
    })# formatted with same names as template.

    return sector_employment_and_wages_by_sub_sector


In [885]:
def unskilled_labor_wages(
    city_names: List[str],
    naics_codes: List[str]=[],
    naics_titles: Optional[str]=[],
    state_name: Optional[str]=None,
)->pd.DataFrame:
    """
    Query BigQuery/JobsEQ data for the unskilled labor salaries by job title table for Company Relocation.

    Args:
        city_names: List of cities requested by user.
        naics_codes: List of NAICS Codes requested for Industries.
        state_name: Optional, state specified by user with 2 digit representation.

    Returns:
        Pandas dataframe representing the unskilled labor salaries by job title table.
    """
    # Create the metro where clause.
    city_where_clause = f"(LOWER(metro) LIKE '%{city_names[0]}%'"
    if len(city_names)>1:
        for city in city_names[1:]:
            city_where_clause = city_where_clause + f" OR LOWER(metro) LIKE '%{city}%'"
    city_where_clause = city_where_clause + ")"
    sql_query = f"SELECT soc, occupation FROM {OCCUPATION_WAGES_2024Q3} WHERE {city_where_clause};"

    # Get Instustry Occupation Data
    query_job = bq_client.query(
        query=sql_query,
    )
    industry_occupations: pd.DataFrame = query_job.to_dataframe()

    # Send to Gemini to decide which to include
    prompt = f""""For the industry sectors {naics_titles}, identify the most relevant occupations and their corresponding Standard Occupational Classification (SOC) codes from the following list.

        List of Occupations and SOC Codes:
        {industry_occupations}

        Instructions:
        1.  Focus specifically on each of the following industries:{naics_titles}.
        2.  Select only the occupations and SOC codes that are directly and significantly applicable to UNSKILL LABOR for the specified industries.
        3.  Exclude occupations that are clearly unrelated or have minimal relevance to all of the requested industries.
        4.  Exclude occupations that would be categoriezed as "skilled" labor, and only returned occupations that would be categorized as "unskilled".
        5.  The number of selected occupations should be around 15.
        6.  Prioritize occupations that are essential for the core functions of the requested industries.
        7.  Output the results as valid JSON, where the key is the SOC Code and the value is the Occupation title (e.g., {{"1234":"Operator"}}.
        8.  Do not output any explanation. Only output the JSON.

        Example Output:
        {{"11-1011": "occupation 1", "11-1021": "occupation 2", ... }}

        """
    response = genai_client.models.generate_content(
        model="gemini-1.5-flash-002",
        # contents="help me create a metro matrix for Houston and Sugar Land",
        # contents="create a hq relocation report for a manufacturing company moving to houston",
        contents=prompt,
        # contents="Can you help me create a specialized city comparison report",
        config=genai_types.GenerateContentConfig(
            system_instruction="I need to find the correct occupations I should be focusig my report on for given industries.",
            temperature=0.2,
            candidate_count=1,
            seed=42,
            response_mime_type = "application/json",
        ),
        )

    response =  response.candidates[0].content.parts[0].text

    # Turn into dict.
    try:
        occupations = json.loads(response)
        if len(occupations) == 0:
            return("Could not find Occupations")
    except json.JSONDecodeError:
        return("Could not find Occupations")

    # Use the selected occupations to create unskilled labor salaries by job title table.
    soc_codes = list(occupations.keys())
    soc_where_clause = f"(soc = '{soc_codes[0]}'"
    for soc in soc_codes[1:]:
        soc_where_clause = soc_where_clause + f" OR soc = '{soc}'"
    soc_where_clause = soc_where_clause + ")"
    unskilled_labor_query = f"SELECT occupation, mean, entry_level, experienced, metro FROM {OCCUPATION_WAGES_2024Q3} WHERE {soc_where_clause} AND {city_where_clause};"

    # Get Instustry Occupation Data
    query_job = bq_client.query(
        query=unskilled_labor_query,
    )
    unskilled_labor: pd.DataFrame = query_job.to_dataframe()

    return unskilled_labor
    

### HQ Relocation Functions

In [886]:
import json

def labor_market_info(
    city_names: List[str],
    naics_codes: List[str]=[],
    state_name: Optional[str]=None,
)->pd.DataFrame:
    """
    Query BigQuery/JobsEQ data for the Area Industry Occupations Breakdown table for HQ Relocation.

    Args:
        city_names: List of cities requested by user. Should only be one city requested.
        naics_codes: List of NAICS Codes requested for Industries.
        state_name: Optional, state specified by user with 2 digit representation.

    Returns:
        Pandas dataframe representing the Area Industry Occupations Breakdown table for HQ Relocation.
    """

    # Get subsector numbers.
    naics_where_clause = f"(naics = {naics_codes[0]}"
    for code in naics_codes[1:]:
        naics_where_clause = naics_where_clause + f" OR naics = {code}"
    naics_where_clause = naics_where_clause + f")"
    sql_query = f"SELECT occupation, empl, avg_hourly_wages_int, avg_ann_wages FROM {INDUSTRY_OCCUPATION_2024Q3} WHERE {naics_where_clause} AND (LOWER(metro) LIKE '%{city_names[0]}%');"
    # Get Instustry Occupation Data
    query_job = bq_client.query(
        query=sql_query,
    )
    labor_market_info: pd.DataFrame = query_job.to_dataframe()

    labor_market_info["avg_hourly_wages_int"] = "$" + labor_market_info["avg_hourly_wages_int"]

    labor_market_info = labor_market_info.rename(columns = {
        "occupation": "Occupational Title",
        "empl": "Total Employees",
        "avg_ann_wages": "Average Annual Salary",
        "avg_hourly_wages_int": "Avgerage Hourly Rate"
    })# formatted with same names as template.

    return labor_market_info
    

## Test Function

In [887]:
empl_and_wages_by_industry_table, unskilled_labor_salaries = jobs_eq_data(
    naics_codes=["325"], #Manufactoring sector
    naics_titles = ["Chemical Manufacturing"],
    city_names=["dallas", "austin"],
    state_name=None,
    economic_request_type="company_relocation"
)

SELECT code, `2022 NAICS US Title` FROM ghp-poc.jobseq.naics_us_code_2022 WHERE (Substring(code, 1, 3) = '325') AND LENGTH(code) = 4;

    SELECT
      SUBSTR(code, 1, 4) AS code,
      SUM(empl) AS sum_total_empl_int,
      SUM(avg_ann_wages_int)/SUM(empl) AS avg_avg_ann_wages_int,
      metro
    FROM
      ghp-poc.jobseq.industry_2024q3_cleaned t
    WHERE
      (LOWER(metro) LIKE '%dallas%' OR LOWER(metro) LIKE '%austin%')
      AND
      SUBSTR(code, 1, 4) IN ('3251', '3252', '3253', '3254', '3255', '3256', '3259') Group by metro, code;


In [888]:
unskilled_labor

,occupation,mean,entry_level,experienced,metro
0,Tool and Die Makers,"$63,000","$44,700","$72,100","Austin-Round Rock-San Marcos, TX"
1,Tool and Die Makers,"$68,100","$43,500","$80,500","Dallas-Fort Worth-Arlington, TX"
2,"Cleaning, Washing, and Metal Pickling Equipmen...","$45,900","$33,700","$52,000","Austin-Round Rock-San Marcos, TX"
3,"Cleaning, Washing, and Metal Pickling Equipmen...","$41,600","$27,600","$48,600","Dallas-Fort Worth-Arlington, TX"
4,Cooling and Freezing Equipment Operators and T...,"$52,700","$31,300","$63,400","Austin-Round Rock-San Marcos, TX"
5,Cooling and Freezing Equipment Operators and T...,"$52,800","$31,400","$63,500","Dallas-Fort Worth-Arlington, TX"
6,Etchers and Engravers,"$35,700","$28,700","$39,200","Austin-Round Rock-San Marcos, TX"
7,Etchers and Engravers,"$38,000","$30,000","$42,000","Dallas-Fort Worth-Arlington, TX"
8,"Production Workers, All Other","$40,800","$30,700","$45,800","Austin-Round Rock-San Marcos, TX"
9,"Production Workers, All Other","$43,500","$32,800","$48,800","Dallas-Fort Worth-Arlington, TX"


In [892]:
empl_and_wages_by_industry_table = jobs_eq_data(
    naics_codes=["325"], #Manufactoring sector
    naics_titles = ["Chemical Manufacturing"],
    city_names=["dallas", "austin"],
    state_name=None,
    economic_request_type="company_relocation"
)

SELECT code, `2022 NAICS US Title` FROM ghp-poc.jobseq.naics_us_code_2022 WHERE (Substring(code, 1, 3) = '325') AND LENGTH(code) = 4;

    SELECT
      SUBSTR(code, 1, 4) AS code,
      SUM(empl) AS sum_total_empl_int,
      SUM(avg_ann_wages_int)/SUM(empl) AS avg_avg_ann_wages_int,
      metro
    FROM
      ghp-poc.jobseq.industry_2024q3_cleaned t
    WHERE
      (LOWER(metro) LIKE '%dallas%' OR LOWER(metro) LIKE '%austin%')
      AND
      SUBSTR(code, 1, 4) IN ('3251', '3252', '3253', '3254', '3255', '3256', '3259') Group by metro, code;


In [893]:
empl_and_wages_by_industry_table # this is the company relocation manufacturing employment and wages by sub industry

(                                             Industry  Employment  \
 0                        Basic Chemical Manufacturing         364   
 1                        Basic Chemical Manufacturing        2519   
 2   Resin, Synthetic Rubber, and Artificial and Sy...        1372   
 3   Resin, Synthetic Rubber, and Artificial and Sy...          33   
 4   Pesticide, Fertilizer, and Other Agricultural ...          20   
 5   Pesticide, Fertilizer, and Other Agricultural ...         506   
 6           Pharmaceutical and Medicine Manufacturing        2489   
 7           Pharmaceutical and Medicine Manufacturing        5809   
 8          Paint, Coating, and Adhesive Manufacturing        2956   
 9          Paint, Coating, and Adhesive Manufacturing          45   
 10  Soap, Cleaning Compound, and Toilet Preparatio...         403   
 11  Soap, Cleaning Compound, and Toilet Preparatio...        3371   
 12  Other Chemical Product and Preparation Manufac...        1869   
 13  Other Chemical 

In [896]:
labor_market_info_table = jobs_eq_data(
    naics_codes=["325"], #Manufactoring sector
    naics_titles = ["Chemical Manufacturing"],
    city_names=["houston"],
    state_name=None,
    economic_request_type="hq_relocation"
)

In [897]:
labor_market_info_table

,Occupational Title,Total Employees,Avgerage Hourly Rate,Average Annual Salary
0,Web and Digital Interface Designers,4,$44.38,"$92,300"
1,"Textile Cutting Machine Setters, Operators, an...",1,$15.72,"$32,700"
2,Editors,1,$52.64,"$109,500"
3,Clinical and Counseling Psychologists,1,$39.57,"$82,300"
4,"Insulation Workers, Floor, Ceiling, and Wall",1,$23.41,"$48,700"
...,...,...,...,...
291,"Mixing and Blending Machine Setters, Operators...",1403,$23.61,"$49,100"
292,Packaging and Filling Machine Operators and Te...,1355,$20.14,"$41,900"
293,First-Line Supervisors of Production and Opera...,2188,$38.99,"$81,100"
294,Chemical Equipment Operators and Tenders,6344,$40,"$83,200"
